> **Release dependency:** the original `clinical_review/` helper package was absent from the supplied source tree. This notebook is retained for documentation but cannot run until those original helpers are restored. See `docs/REPRODUCIBILITY.md`. No completed reviewer data are distributed.

# BR07A — Import completed independent ECG interpretations

Run after BR07 has created the calibration package and after the clinician has completed `reference_cases_completed.json`. The default input is the exact uploaded Biowulf file. This notebook validates and imports the human entries directly; it does **not** call a script, load a generator, rerender images, or modify the protected `reference_cases.json` template.

All cases are validated before any reference draft is written. The reviewed-image hashes must match the frozen package. Existing identical drafts are skipped, allowing a restart; conflicting assessments stop the import. Independent interpretations already submitted in BR08 cannot be overwritten.

Run the code cells in order in your CPU evaluation/review kernel. No other new Python helper is needed. The existing `common.py` and `clinical_review/workflow.py` must remain unchanged.

In [ ]:
from pathlib import Path
import sys
import json
import hashlib
import os
from datetime import datetime, timezone

EXP = Path('/path/to/repository/experiment02')
REVIEW_DIR = Path(
    '/path/to/repository/outputs/real_run_001/'
    'experiment02_v1/context_fit/v1/internal_val/clinical_review/internal_val/calibration_v1'
)
INPUT_PATH = REVIEW_DIR / 'PUBLIC' / 'reference_cases_completed.json'
# For another cohort/version, change REVIEW_DIR to its existing BR07 package.
if not (EXP / 'common.py').is_file():
    raise FileNotFoundError('Set EXP to the experiment02 folder containing common.py')
sys.path.insert(0, str(EXP))
from common import read, sha, immutable, lock
from clinical_review.workflow import _check, save_reference, progress

if not (REVIEW_DIR / 'PRIVATE' / 'protocol.json').is_file():
    raise FileNotFoundError('Existing BR07 package not found: ' + str(REVIEW_DIR))
if not INPUT_PATH.is_file():
    raise FileNotFoundError('Completed reference JSON not found: ' + str(INPUT_PATH))
print('Kernel:', sys.executable)
print('Review package:', REVIEW_DIR)
print('Completed human reviews:', INPUT_PATH)
print('Current progress:', progress(REVIEW_DIR))


## Validate the entire batch

This checks the original package signatures, exact case set, reviewer identity, completed fields and source-image checksums. It also checks that no conflicting reference drafts already exist. It does not display model identities or generated reports.

Use the upload-ready `reference_cases_completed.json`, which includes `source_image_sha256` for each locally reviewed image. Do not calculate replacement hashes on Biowulf to make a mismatching image pass. If you already replaced the protected template, this cell will stop; preserve the files and resolve the original template mismatch rather than changing its recorded checksum.

In [ ]:
REFERENCE_FIELDS = ('case_id', 'reviewer_id', 'interpretable',
                    'independent_ecg_interpretation', 'limitations')

def validate_completed_references():
    # Native verification preserves the signed template, source and image checks.
    protocol = _check(REVIEW_DIR)
    if ((REVIEW_DIR / 'PRIVATE/references_submitted.json').exists()
            or (REVIEW_DIR / 'PUBLIC/reports.json').exists()):
        raise ValueError('Independent references are already submitted/reports released. '
                         'Do not reimport; continue the existing BR08 workflow.')
    payload = INPUT_PATH.read_bytes()
    source_sha = hashlib.sha256(payload).hexdigest()
    rows = json.loads(payload)
    if not isinstance(rows, list) or not rows or not all(isinstance(r, dict) for r in rows):
        raise ValueError('Completed references must be a nonempty JSON list of case objects')
    templates = read(REVIEW_DIR / 'PUBLIC/reference_cases.json')
    expected = {r['case_id']: r for r in templates}
    ids = [r.get('case_id') for r in rows]
    if (any(not isinstance(cid, str) for cid in ids)
            or len(set(ids)) != len(ids) or set(ids) != set(expected)):
        raise ValueError('Case identifiers must be unique and match every case in this BR07 package')
    snapshot = REVIEW_DIR / 'PRIVATE/imported_reference_cases.json'
    if snapshot.exists() and sha(snapshot) != source_sha:
        raise ValueError('A different completed JSON was previously imported. '
                         'Preserve it and review the discrepancy explicitly.')
    records, existing_ids = [], set()
    for row in rows:
        cid = row['case_id']
        if row.get('reviewer_id') != protocol['reviewer_id']:
            raise ValueError('Reviewer identity mismatch: ' + cid)
        if type(row.get('interpretable')) is not bool:
            raise ValueError('interpretable must be true or false: ' + cid)
        interpretation = row.get('independent_ecg_interpretation')
        if not isinstance(interpretation, str) or not interpretation.strip():
            raise ValueError('Missing independent ECG interpretation: ' + cid)
        if not isinstance(row.get('limitations'), str):
            raise ValueError('limitations must be a string: ' + cid)
        if row.get('image') != expected[cid]['image']:
            raise ValueError('Case/image mapping differs from the package: ' + cid)
        image = REVIEW_DIR / 'PUBLIC' / expected[cid]['image']
        if (not row.get('source_image_sha256') or not image.is_file()
                or sha(image) != row['source_image_sha256']):
            raise ValueError('Locally reviewed image does not match the frozen package: ' + cid)
        record = {k: row[k] for k in REFERENCE_FIELDS}
        draft = REVIEW_DIR / 'PUBLIC/reference_drafts' / (cid + '.json')
        if draft.exists():
            saved = read(draft)
            if any(saved.get(k) != record[k] for k in REFERENCE_FIELDS):
                raise ValueError('Existing clinician assessment differs: ' + cid + '. No drafts overwritten.')
            existing_ids.add(cid)
        records.append(record)
    return protocol, records, existing_ids, payload, source_sha

protocol, records, existing_ids, _, source_sha = validate_completed_references()
print('Validated cases:', len(records))
print('Reviewer:', protocol['reviewer_id'])
print('Matching drafts already saved:', len(existing_ids))
print('New drafts to import:', len(records) - len(existing_ids))
for row in records:
    print(row['case_id'], '| interpretable:', row['interpretable'],
          '| interpretation characters:', len(row['independent_ecg_interpretation']))


## Import the validated clinician entries

Run the following cell to save the completed independent interpretations. It validates the batch again under a single-writer lock, saves an exact private copy of the uploaded JSON, and uses the existing native `save_reference` function for every new draft. No clinical content is generated or rewritten.

The import is resumable before BR08 submission: rerun this notebook with the same file and already matching drafts are preserved. If interrupted, only unsaved entries remain. The source JSON, protected template, images and generator files are not changed.

In [ ]:
with lock(REVIEW_DIR / 'reference_import.lock'):
    protocol, records, existing_ids, payload, source_sha = validate_completed_references()
    snapshot = REVIEW_DIR / 'PRIVATE/imported_reference_cases.json'
    if not snapshot.exists():
        temporary = snapshot.with_name(snapshot.name + '.tmp')
        with temporary.open('wb') as stream:
            stream.write(payload)
            stream.flush()
            os.fsync(stream.fileno())
        temporary.replace(snapshot)
    imported = 0
    for row in records:
        if row['case_id'] in existing_ids:
            continue
        save_reference(REVIEW_DIR, row['case_id'],
            reviewer_id=row['reviewer_id'], interpretable=row['interpretable'],
            independent_ecg_interpretation=row['independent_ecg_interpretation'],
            limitations=row['limitations'])
        imported += 1
    log_path = REVIEW_DIR / 'PRIVATE/reference_notebook_import.json'
    imported_at = (read(log_path)['imported_at_utc'] if log_path.exists()
                   else datetime.now(timezone.utc).isoformat())
    immutable(log_path, dict(
        workflow='BR07A_import_completed_reference_reviews_v1',
        imported_at_utc=imported_at, source_sha256=source_sha,
        cases=[r['case_id'] for r in records], reviewer_id=protocol['reviewer_id'],
        package_protocol_sha256=sha(REVIEW_DIR / 'PRIVATE/protocol.json'),
        image_identity_verified=True, protected_template_modified=False))
print('New interpretations imported:', imported)
print('Previously saved identical interpretations:', len(existing_ids))
print('Progress:', progress(REVIEW_DIR))


In [ ]:
status = progress(REVIEW_DIR)
if status['references_saved'] != status['reference_cases']:
    raise ValueError('Some independent interpretations remain unsaved; do not release reports yet.')
print('READY FOR BR08:', status)
print('Protected template remains unchanged:', REVIEW_DIR / 'PUBLIC/reference_cases.json')
print('Saved assessments:', REVIEW_DIR / 'PUBLIC/reference_drafts')
print('Continue to BR08 using the SAME review directory and REVIEW_VERSION.')


## Next step

For this package the expected result is **10 reference cases, 10 references saved and 0 reports released**. You do not need to re-enter those interpretations in BR07.

Proceed to **BR08**. Its release cell freezes the independent interpretations and releases the **40 anonymous generated reports** for initial assessment and subsequent same-clinician adjudication. Do not edit the independent references after report release. Run BR09 after report assessment and required adjudications are complete.